In [11]:
import pandas as pd
from bert_score import score
from evaluate import load
import os
# !pip install bert-score evaluate torch

# Load evaluation metric once
bert_metric = load("bertscore")
reference_file = 'dataset_clean_fixed.csv'

def run_bertscore_eval(results_file, output_name, task_label):
    """
    Unified function to load results, align with ground truth,
    calculate BERTScore, and save the results.
    """
    print(f"Starting Evaluation: {task_label}")

    # Check if files exist
    if not os.path.exists(results_file) or not os.path.exists(reference_file):
        print(f"Error: Missing {results_file} or {reference_file}")
        return None

    # Load Results: Comma separated
    results_df = pd.read_csv(results_file, sep = ',', engine = 'python', on_bad_lines = 'skip')

    # Load Reference: Semicolon separated (Crucial fix!)
    reference_df = pd.read_csv(reference_file, sep = ';')

    # Merge on 'id' to ensure samples are correctly paired
    df = results_df.merge(reference_df, on = "id")

    # Drop rows where either answer or ground truth is missing
    df = df.dropna(subset = ["answer", "correct_answer"])

    if df.empty:
        print(f"Warning: No matching data for {task_label}. Check CSV 'id' columns.")
        return None

    predictions = df['answer'].fillna("").tolist()
    references = df['correct_answer'].tolist()

    print(f"Evaluating {len(predictions)} samples...")

    # Compute BERTScore for German (de)
    # rescale_with_baseline = True makes scores more interpretable
    P, R, F1 = score(
        predictions,
        references,
        lang = "de",
        rescale_with_baseline = True,
        verbose = False
    )

    # Extract mean values for the final report
    # We use .item() to convert single-value tensors to floats
    mean_p = P.mean().item()
    mean_r = R.mean().item()
    mean_f1 = F1.mean().item()

    # Add individual scores back to the dataframe for analysis
    df['bert_p'] = P.tolist()
    df['bert_r'] = R.tolist()
    df['bert_f1'] = F1.tolist()

    # Also attach the overall average to every row for easy lookup
    df['avg_f1_score'] = mean_f1

    # Output Final Report to Console
    print(f"Results for {task_label}:")
    print(f"Precision: {mean_p:.4f}")
    print(f"Recall: {mean_r:.4f}")
    print(f"F1 Score: {mean_f1:.4f}\n")

    # Save the full results including scores to a new CSV
    df.to_csv(output_name, index = False)

    return df

# Execute for all three experiments

# 1. Zero-Shot Inference
inference_eval = run_bertscore_eval(
    'inference_results.csv',
    'inference_evaluated_results.csv',
    'Zero-Shot Inference'
)

# 2. Fine-Tuned Model
finetuned_eval = run_bertscore_eval(
    'fine_tuned_results.csv',
    'fine_tuned_evaluated_results.csv',
    'Fine-Tuned Model'
)

# 3. RAG System
rag_eval = run_bertscore_eval(
    'rag_results.csv',
    'rag_evaluated_results.csv',
    'RAG System'
)

Starting Evaluation: Zero-Shot Inference
Evaluating 643 samples...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results for Zero-Shot Inference:
Precision: 0.0537
Recall: 0.3087
F1 Score: 0.1718

Starting Evaluation: Fine-Tuned Model
Evaluating 643 samples...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results for Fine-Tuned Model:
Precision: 0.3260
Recall: 0.2494
F1 Score: 0.2855

Starting Evaluation: RAG System
Evaluating 643 samples...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Results for RAG System:
Precision: 0.2685
Recall: 0.1718
F1 Score: 0.2180

